# 异常处理进阶 · 练习

### 1、try/except/else/finally 完整结构

In [ ]:
from contextlib import contextmanager, suppress


# 写一个函数 safe_convert(text)，尝试把 text 转成 int：
# - 转换失败（ValueError），打印 "转换失败：xxx 不是数字"
# - 转换成功，在 else 块里打印 "转换成功：结果是 xxx"
# - 不管成功失败，finally 都打印 "处理结束"
#
# 测试用例：
# safe_convert("123")
# safe_convert("abc")
# 期望输出：
# 转换成功：结果是 123
# 处理结束
# 转换失败：abc 不是数字
# 处理结束

# 在这里写你的代码

def safe_convert(text:str):
    try:
        value = int(text)
    except ValueError as e:
        print(f"转换失败：{text} 不是数字")
    else:
        print(f"转换成功：结果是 {value}")
    finally:
        print("处理结束")

safe_convert("123")
safe_convert("abc")


### 2、异常捕获顺序与类型层级

In [ ]:
# 写一个函数 safe_divide(a, b)，计算 a / b：
# - 如果 b 是 0，捕获 ZeroDivisionError，打印 "除数不能为 0"
# - 如果 a 或 b 不是数字（比如传字符串导致 TypeError），打印 "参数类型错误"
# 要求把这两个 except 分开写（不要合并成一个元组），体会"具体异常写在前面"的顺序要求
#
# 测试用例：
# safe_divide(10, 0)        # 除数不能为 0
# safe_divide(10, "a")      # 参数类型错误
# safe_divide(10, 2)        # 打印 5.0

# 在这里写你的代码

def safe_divide(a, b):
    try:
        value = a / b
        print(value)
    except ZeroDivisionError as e:
        print("除数不能为 0")
    except TypeError as e:
        print("参数类型错误")

safe_divide(10, 0)
safe_divide(10, "a")
safe_divide(10, 2)

### 3、主动抛出异常：raise

In [ ]:
# 1. 写一个函数 set_age(age)：如果 age < 0 或 age > 150，主动 raise ValueError，
#    异常信息里带上具体的 age 值，比如 "年龄不合法：200"
# 2. 写一个函数 set_age_with_log(age)：内部调用 set_age(age)，用 try/except 捕获到异常后
#    先打印一行 "记录日志：年龄设置失败"，再用不带参数的 raise 把异常原样重新抛出
# 3. 在最外层用 try/except 调用 set_age_with_log(200)，验证：
#    - "记录日志：年龄设置失败" 打印了
#    - 外层依然捕获到了同一个 ValueError，并打印异常信息

# 在这里写你的代码

def set_age(age):
    if age < 0 or age > 150 :
        raise ValueError(f"年龄不合法{age}")

def set_age_with_log(age):
    try:
        set_age(age)
    except ValueError as e:
        print("记录日志：年龄设置失败")
        raise

try:
    set_age_with_log(200)
except ValueError as e:
    print(e)

### 4、自定义异常类

In [15]:
# 电商下单时，如果库存不足要抛出自定义异常 OutOfStockError
# 要求：
# 1. 异常带 product_name、requested、available 三个属性
# 2. 异常信息格式："库存不足：商品「鼠标」库存剩 3 件，需要 5 件"
# 写一个函数 place_order(product_name, requested, available)：
#   requested > available 时抛出 OutOfStockError，否则返回字符串 "下单成功"
#
# 测试用例：
# place_order("鼠标", 5, 3)   # 抛出异常，捕获后打印异常信息和 product_name/requested/available 三个属性
# place_order("鼠标", 2, 3)   # 返回 "下单成功"

# 在这里写你的代码

class OutOfStockError(Exception):
    def __init__(self,product_name,requested,available):
        self.product_name = product_name
        self.requested = requested
        self.available = available
        super().__init__(f"库存不足：商品「{product_name}」库存剩 {available} 件，需要 {requested} 件")

def place_order(product_name,requested,available):
    if requested > available:
        raise OutOfStockError(product_name,requested,available)
    return "下单成功"

try:
    place_order("鼠标", 5, 3)
except OutOfStockError as e:
    print(e)
    print(e.product_name, e.requested, e.available)

print(place_order("鼠标", 2, 3))

库存不足：商品「鼠标」库存剩 3 件，需要 5 件
鼠标 5 3
下单成功


### 5、异常链 raise...from

In [18]:
# 写一个函数 parse_user_data(text)，内部用 json.loads(text) 解析文本
# 如果解析失败（捕获到 json.JSONDecodeError），抛出自定义异常 UserDataError("用户数据格式错误")
# 要求用 raise ... from 重新抛出，保留原始异常信息
#
# 测试用例：
# parse_user_data('{"name": "Alice"}')   # 正常返回解析结果
# parse_user_data("not a json")          # 抛出 UserDataError，捕获后打印：
#   异常信息（用户数据格式错误）
#   e.__cause__（原始的 JSONDecodeError）

# 在这里写你的代码

class UserDataError(Exception):
    def __init__(self):
        super().__init__("用户数据格式错误")

import json
def parse_user_data(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise UserDataError() from e

print(parse_user_data('{"name": "Alice"}'))
try:
    print(parse_user_data('not a json'))
except UserDataError as e:
    print(e)
    print(e.__cause__)

{'name': 'Alice'}
用户数据格式错误
Expecting value: line 1 column 1 (char 0)


### 6、with 语句与上下文管理器协议（__exit__ 返回值的效果）

In [21]:
# 写一个上下文管理器类 NoOpManager：
# - __enter__：打印 "进入"，返回 self
# - __exit__：打印 "退出"，并且 return True（表示吞掉异常）
#
# 用 with NoOpManager(): 包一段代码，块内故意写一行 1/0 触发异常
# 验证：程序不会因为这个异常崩溃（异常被吞掉了），但 "进入" 和 "退出" 都正常打印
# 对比第 7 题的 DBConnection（__exit__ 里 return False，不吞异常）：
# 体会 __exit__ 返回值不同，异常传播的结果完全不同

# 在这里写你的代码

class NoOpManager:
    def __enter__(self):
        print("进入")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("退出")
        return True

with NoOpManager() as f :
    a = 1/0

进入
退出


### 7、自定义上下文管理器类

In [23]:
# 写一个上下文管理器类 DBConnection，模拟数据库连接：
# - __enter__：打印 "连接数据库"，返回 self
# - __exit__：打印 "关闭数据库连接"，不吞异常（return False）
#
# 用 with DBConnection() as conn: 包一段代码，块内故意写一行 1/0 触发异常
# 外层用 try/except 包住整个 with，验证两件事：
# 1. "连接数据库" 和 "关闭数据库连接" 都打印了（证明 __exit__ 一定会执行）
# 2. 异常正常被外层 except 捕获到（证明 __exit__ 没有吞掉异常）

# 在这里写你的代码

class DBConnection:
    def __enter__(self):
        print("连接数据库")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("关闭数据库连接")
        return False
try:
    with DBConnection() as conn:
        a = 1/0
except ZeroDivisionError as e:
    print(e)

链接数据库
关闭数据库连接
division by zero


### 8、contextlib.contextmanager 简化版

In [26]:
# 用 @contextmanager 装饰器，把第 7 题的 DBConnection 改写成函数形式的上下文管理器 db_connection()
# 效果要求一样：进入时打印 "连接数据库"，退出时打印 "关闭数据库连接"
# 块内出错（比如还是写 1/0）时，也要保证退出语句照常打印，并且异常正常往外抛、能被外层捕获到

# 在这里写你的代码

from contextlib import contextmanager

@contextmanager
def db_connection():
    print("连接数据库")
    try:
        yield
    finally:
        print("关闭数据库连接")

try:
    with db_connection() :
        a = 1/0
except ZeroDivisionError as e:
    print(e)

连接数据库
关闭数据库连接
division by zero


### 9、contextlib.suppress

In [27]:
# 有一个清理函数，会尝试删除一个文件：os.remove(path)
# 文件不存在时会报 FileNotFoundError
# 用 contextlib.suppress 简化这个操作（不要写 try/except），
# 不管文件存不存在，删除完都继续往下打印 "清理完成"
#
# 测试用例：
# cleanup("not_exist_file.txt")   # 文件不存在也不报错，照常打印 "清理完成"

# 在这里写你的代码
from contextlib import suppress
import os
def cleanup(path:str):
    with suppress(FileNotFoundError):
        os.remove(path)
    print("清理完成")
cleanup("not_exist_file.txt")

清理完成
